In [0]:
%pip install google-cloud-bigquery google-cloud-bigquery-storage db-dtypes
dbutils.library.restartPython()


In [0]:
from google.cloud import bigquery
from google.oauth2 import service_account
import pandas as pd
import os

In [0]:

# Json
key_json_path = os.path.abspath(os.path.join("..", "bqueryconnect.json"))

project_id = 'free-407612'
dataset = 'business_analysis'
# Path to your service account key (uploaded to Databricks)
credentials = service_account.Credentials.from_service_account_file(key_json_path)

# Initialize BigQuery client
client = bigquery.Client(credentials=credentials, project=project_id)

# Define your dataset
dataset_id = f"{project_id}.{dataset}"

# Function to load Delta table to BigQuery
def load_to_bigquery(spark_table_name, bq_table_name):
    # Read from Delta - ADD BACKTICKS for proper table reference
    df_spark = spark.sql(f"SELECT * FROM {spark_table_name}")
    
    # Convert to Pandas (for manageable datasets)
    df_pandas = df_spark.toPandas()
    
    # Define destination
    table_id = f"{dataset_id}.{bq_table_name}"
    
    # Load to BigQuery
    job_config = bigquery.LoadJobConfig(
        write_disposition="WRITE_TRUNCATE",  # Overwrite table
        autodetect=True
    )
    
    job = client.load_table_from_dataframe(
        df_pandas, table_id, job_config=job_config
    )
    
    job.result()  # Wait for completion
    print(f"✅ Loaded {spark_table_name} → {table_id}")
    print(f"   Rows: {job.output_rows}")



In [0]:
# gold_unique_products
load_to_bigquery("bi_analytics.gold.gold_unique_products", "gold_unique_products")
# gold_contribucion_p_institucion
load_to_bigquery("bi_analytics.gold.gold_contribucion_p_institucion", "gold_contribucion_p_institucion")
# gold_maximum_historics
load_to_bigquery("bi_analytics.gold.gold_maximum_historics", "gold_maximum_historics")
# gold_maximum_historics
load_to_bigquery("bi_analytics.gold.purchases_date_range", "purchases_date_range")
